In [3]:
import os

# Create the data/raw directory if it doesn't exist
os.makedirs('data/raw/', exist_ok=True)

# Unzip the archive to extract the CSV files
!unzip -o '/content/archive (2).zip' -d .

# Move the files
!mv train.csv data/raw/
!mv test.csv data/raw/

Archive:  /content/archive (2).zip
  inflating: ./test.csv              
  inflating: ./train.csv             


In [5]:
import pandas
import numpy
import sklearn
import joblib
import nltk

In [7]:
RAW_TRAIN_PATH = "data/raw/train.csv"
RAW_TEST_PATH = "data/raw/test.csv"

PROCESSED_TRAIN_PATH = "data/processed/train_clean.csv"
PROCESSED_TEST_PATH = "data/processed/test_clean.csv"

MODEL_PATH = "models/news_classifier.pkl"
RESULT_PATH = "results/metrics.txt"


In [9]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

def preprocess():
    train = pd.read_csv(RAW_TRAIN_PATH)
    test = pd.read_csv(RAW_TEST_PATH)

    train.columns = ["label", "title", "description"]
    test.columns = ["label", "title", "description"]

    train["text"] = train["title"] + " " + train["description"]
    test["text"] = test["title"] + " " + test["description"]

    train["text"] = train["text"].apply(clean_text)
    test["text"] = test["text"].apply(clean_text)

    train = train[["text", "label"]]
    test = test[["text", "label"]]

    train.to_csv(PROCESSED_TRAIN_PATH, index=False)
    test.to_csv(PROCESSED_TEST_PATH, index=False)

    print("Preprocessing done")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

def feature_engineering():
    train = pd.read_csv(PROCESSED_TRAIN_PATH)
    test = pd.read_csv(PROCESSED_TEST_PATH)

    vectorizer = TfidfVectorizer(max_features=5000)

    X_train = vectorizer.fit_transform(train["text"])
    X_test = vectorizer.transform(test["text"])

    joblib.dump(vectorizer, "models/vectorizer.pkl")

    return X_train, X_test, train["label"], test["label"]

In [13]:
from sklearn.naive_bayes import MultinomialNB
import joblib

def train_model(X_train, y_train):
    model = MultinomialNB()
    model.fit(X_train, y_train)

    joblib.dump(model, MODEL_PATH)
    print("Model trained and saved")

In [15]:
import joblib
from sklearn.metrics import accuracy_score, confusion_matrix

def evaluate(X_test, y_test):
    model = joblib.load(MODEL_PATH)

    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)

    with open(RESULT_PATH, "w") as f:
        f.write(f"Accuracy: {acc}\n")
        f.write("Confusion Matrix:\n")
        f.write(str(cm))

    print("Evaluation complete")
    print("Accuracy:", acc)

In [18]:
import os

def run_pipeline():
    # Create necessary output directories
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('models', exist_ok=True)
    os.makedirs('results', exist_ok=True)

    preprocess()
    X_train, X_test, y_train, y_test = feature_engineering()
    train_model(X_train, y_train)
    evaluate(X_test, y_test)

if __name__ == "__main__":
    run_pipeline()

Preprocessing done
Model trained and saved
Evaluation complete
Accuracy: 0.8896052631578948
